# Boosting

This notebook implements a **Gradient Boosting** approach to fake news detection, following the optimization problem defined in the project specification. The objective minimises a weighted, category-balanced cross-entropy loss with regularisation:

$$Z = -\frac{1}{N}\sum_{i=1}^{N} \alpha_{c_i}\left[w_1 y_i \log(F(x_i)) + w_0(1-y_i)\log(1-F(x_i))\right] + \lambda\Omega(\Theta)$$

where $F(x) = \sum_{m=1}^{M} \beta_m f_m(x)$ is the boosting ensemble.

Project Environment: Run the code cell below to auto-detect data ingestion script. <br>
Stand-alone Environment : Skip the code cell and upload data ingestion script from github.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/project/COMP3608PROJECT')

Mounted at /content/drive


#Data Ingestion
Load the unified fake-news DataFrame using the shared `ingest_data` script. This combines three Kaggle datasets (`bhavikjikadara`, `mahdimashayekhi`, `shawkyelgendy`) and returns a DataFrame with columns: `title`, `text`, `label`, `category`, `dataset`. Basic preprocessing (null-text removal, deduplication, category normalisation) is applied inside the loader.

In [3]:
from ingest_data import load_datasets
df = load_datasets()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...


100%|██████████| 18.1M/18.1M [00:00<00:00, 134MB/s]

Extracting zip of true.csv...


[bhavik] Loaded 'true.csv': 21417 rows


100%|██████████| 22.9M/22.9M [00:00<00:00, 143MB/s]

...

100%|██████████| 32.8M/32.8M [00:00<00:00, 57.2MB/s]


[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...


100%|██████████| 1.08M/1.08M [00:00<00:00, 78.4MB/s]

Extracting zip of real.csv...
[shawky] Loaded 'real.csv': 21869 rows


Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'fake.csv': 19999 rows

Dropped 648 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
------------------------------------------------------------
Total rows: 99,468
Fake (0): 47,089
Real (1): 52,379

Rows per source:
shawkyelgendy          40,824
bhavikjikadara         38,644
mahdimashayekhi        20,000

Categories:
  Sports                 43,691
  Politics               21,635
  News                   19,811
  Health                 2,922
  Entertainment          2,889
  Technology             2,882
  Business               2,849
  Science                2,789
------------------------------------------------------------


#Imports

All third-party libraries that are used in this notebook are imported here for ease of reproducibility and clarity

In [4]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack, csr_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

#Text Cleaning Pipeline

Raw text from multiple sources contains noise: HTML entities, URLs, Twitter handles and other characteristics. the `clean text` essentially is normalizing each entry before vectorization.

The `title` and `text` columns are cleaned independently, then concatenated into a single `combined_text` column — this preserves the signal in headlines without polluting the body text distribution.

In [5]:
# Normalise a raw text string for TF-IDF vectorization
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)   # URLs
    s = re.sub(r'@\w+', ' ', s)                     # Twitter handles
    s = re.sub(r'<[^>]+>', ' ', s)                  # HTML tags
    s = re.sub(r'[^a-z\s]', ' ', s)                 # Non-alpha
    s = re.sub(r'\s+', ' ', s).strip()              # Whitespace
    return s

df['clean_title'] = df['title'].apply(clean_text)
df['clean_text']  = df['text'].apply(clean_text)

# Combine title + text; title is repeated to upweight headline signal
df['combined_text'] = df['clean_title'] + ' ' + df['clean_text']

print(f"Sample cleaned text:\n{df['combined_text'].iloc[0][:300]}")

Sample cleaned text:
as u s budget fight looms republicans flip their fiscal script washington reuters the head of a conservative republican faction in the u s congress who voted this month for a huge expansion of the national debt to pay for tax cuts called himself a fiscal conservative on sunday and urged budget restr


#Term Frequency + Inverse Document Frequency (TF+IDF) Vectorization and Feature Concatenation

The feature vector for each sample is defined by the constraint:

$$x_i = \phi(title_i,\ text_i,\ category_i,\ dataset_i)$$

**Sub-tasks:**
- Fit a TF-IDF vectoriser on `combined_text` (unigrams + bigrams, capped at 50,000 features, sublinear TF scaling).
- One-hot encode the `category` and `dataset` categorical columns (Transforming categorical columns into binary columns once per category).
- Horizontally stack the sparse TF-IDF matrix with the dense categorical features into a single sparse feature matrix `X`.

##TF-IDF Vectorization

In [7]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(
  max_features = 50000, # Limits features to 50,000 most frequent tokens, preventing runaway memory usage
  ngram_range = (1, 2),
  sublinear_tf = True, # Applies log(1+count) instead of using the raw count of the words
  min_df = 3, # Ignores words that appears in fewer than 3 documents since they are too rare to generalize
  strip_accents = 'unicode', # normalizes characters (ie: characters with accents or other special symbols to them)
)
X_tfidf = tfidf.fit_transform(df['combined_text']) # fit_transform learns the vocab from all the articles (fit) and then converts tehm into TF_IDF vector (Transform)


##One-Hot Encode Categorical Features

In [8]:
cat_dummies  = pd.get_dummies(df['category'], prefix='cat').astype(float)
data_dummies = pd.get_dummies(df['dataset'],  prefix='ds').astype(float)

X_cat  = csr_matrix(cat_dummies.values)
X_data = csr_matrix(data_dummies.values)

print(f"Category features : {X_cat.shape[1]}")
print(f"Dataset features  : {X_data.shape[1]}")

Category features : 8
Dataset features  : 3


##Concatenate all features

In [11]:
X = hstack([X_tfidf, X_cat, X_data]) # Horizontally stacks the three feature matricies side by side thus creating a wide matrix 'X'
y = df['label'].values # Stores the labels in an array of fake and real

print(f"Final feature matrix shape : {X.shape}")
print(f"Label distribution:- Fake (0): {(y==0).sum():,}  Real (1): {(y==1).sum():,}")

Final feature matrix shape : (99468, 50011)
Label distribution:- Fake (0): 47,089  Real (1): 52,379
